######## 기존 버전 삭제 후 재설치 (안전한 교체를 위해)  
pip uninstall torch torchvision torchaudio -y
  
######## CUDA 12.1 버전용 PyTorch 설치  
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

pip install timm opencv-python matplotlib tqdm requests

In [1]:
import os
import cv2
import torch
import numpy as np
from models.network_swinir import SwinIR

c:\Users\thlee\AppData\Local\Programs\Python\Python311\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 장치 이름: {torch.cuda.get_device_name(0)}")
else:
    print("GPU를 인식할 수 없습니다. CPU 버전 PyTorch가 설치된 것 같습니다.")

PyTorch 버전: 2.5.1+cu121
CUDA 사용 가능 여부: True
GPU 장치 이름: NVIDIA GeForce GTX 1650 Ti with Max-Q Design


In [ ]:
# 1. 환경 설정 및 경로 지정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = '003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth'
input_image_path = '1_iiFWZ_1HD4BMVZvg6nT6mw.webp'
output_image_path = '1_iiFWZ_1HD4BMVZvg6nT6mw_up.png'

print(f"현재 사용 중인 장치: {device}")

# 수정된 모델 초기화 (upsampler를 'nearest+conv'로 변경)
model = SwinIR(
    upscale=4, 
    in_chans=3, 
    img_size=64, 
    window_size=8,
    img_range=1., 
    depths=[6, 6, 6, 6, 6, 6], 
    embed_dim=180, 
    num_heads=[6, 6, 6, 6, 6, 6], 
    mlp_ratio=2, 
    upsampler='nearest+conv', # 이 부분이 에러를 해결하는 핵심입니다!
    resi_connection='1conv'
)

# 3. 가중치 로드
pretrained_model = torch.load(model_path, map_location='cpu')
param_key = 'params' if 'params' in pretrained_model else 'params_ema' if 'params_ema' in pretrained_model else None

if param_key:
    model.load_state_dict(pretrained_model[param_key], strict=True)
else:
    model.load_state_dict(pretrained_model, strict=True)

model.to(device)
model.eval()
print("모델 로드 성공!")

# 4. 이미지 전처리
img_lq = cv2.imread(input_image_path, cv2.IMREAD_COLOR).astype(np.float32) / 255.
img_lq = np.transpose(img_lq[:, :, [2, 1, 0]], (2, 0, 1)) # BGR to RGB, HWC to CHW
img_lq = torch.from_numpy(img_lq).float().unsqueeze(0).to(device) # Tensor 변환 및 배치 차원 추가

# 5. 추론 (Inference)
print("업스케일링 진행 중...")
with torch.no_grad():
    # 이미지 크기가 window_size(8)의 배수가 아닐 경우를 대비한 패딩 (필요시 자동 적용)
    _, _, h_old, w_old = img_lq.size()
    h_pad = (h_old // 8 + 1) * 8 - h_old if h_old % 8 != 0 else 0
    w_pad = (w_old // 8 + 1) * 8 - w_old if w_old % 8 != 0 else 0
    img_lq = torch.nn.functional.pad(img_lq, (0, w_pad, 0, h_pad), mode='reflect')
    
    output = model(img_lq)
    
    # 패딩 제거
    output = output[..., :h_old * 4, :w_old * 4]

# 6. 결과 후처리 및 저장
output = output.data.squeeze().float().cpu().clamp_(0, 1).numpy()
output = np.transpose(output[[2, 1, 0], :, :], (1, 2, 0)) # RGB to BGR, CHW to HWC
output = (output * 255.0).round().astype(np.uint8)

cv2.imwrite(output_image_path, output)
print(f"저장 완료: {output_image_path}")s

현재 사용 중인 장치: cuda


C:\Users\thlee\AppData\Local\Temp\ipykernel_26980\732171747.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_model = torch.load(model_path, map_location='cpu'

모델 로드 성공!
업스케일링 진행 중...
저장 완료: 1_iiFWZ_1HD4BMVZvg6nT6mw_up.png
